
### Part 2: Multi-Source RAG with Routing (100 points)

Build a RAG system that intelligently routes queries to different data sources based on the question type.

**The Data:**

The `data/` folder contains two types of data sources simulating an e-commerce analytics scenario:

```
data/
├── structured/
│   └── daily_sales.csv      # 1000 rows of sales transactions
└── unstructured/
    ├── ELEC001_product_page.txt   # Product descriptions & reviews
    ├── HOME003_product_page.txt
    ├── SPRT001_product_page.txt
    ├── BEAU001_product_page.txt
    ├── CLTH001_product_page.txt
    ├── BOOK001_product_page.txt
    ├── TOYS001_product_page.txt
    ├── OFFC001_product_page.txt
    ├── PETS001_product_page.txt
    └── FOOD001_product_page.txt
```

**Structured Data (CSV):**
- 1000 daily sales records across 90 days (Oct-Dec 2024)
- 35 products across 10 categories
- Columns: `date, product_id, product_name, category, units_sold, unit_price, total_revenue, region`
- Regions: North, South, East, West, Central

**Unstructured Data (Text):**
- 10 product pages with detailed descriptions, specifications, and customer reviews
- Similar to what you'd find on an e-commerce website
- Includes product features, technical specs, and 5 customer reviews per product

**System Architecture:**

```
┌─────────────┐     ┌──────────────────┐     ┌─────────────────┐     ┌─────────────┐
│ User Query  │────>│  Query Router    │────>│  Data Source(s) │────>│ LLM Answer  │
│             │     │ (classify query) │     │  CSV | Text     │     │ with Context│
└─────────────┘     └──────────────────┘     └─────────────────┘     └─────────────┘
                            │
                            ├── Analytical/numerical → CSV (use bash: grep, awk, cut)
                            ├── Product details/reviews → Unstructured text
                            └── Both → Combine sources
```

**Test Questions:**

Your system should be able to answer the following questions:

| # | Question | Source Required |
|---|----------|-----------------|
| 1 | "What was the total revenue for Electronics category in December 2024?" | CSV Only |
| 2 | "Which region had the highest sales volume?" | CSV Only |
| 3 | "What are the key features of the Wireless Bluetooth Headphones?" | Text Only |
| 4 | "What do customers say about the Air Fryer's ease of cleaning?" | Text Only |
| 5 | "Which product has the best customer reviews and how well is it selling?" | Both (Text + CSV) |
| 6 | "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?" | Both (Text + CSV) |

*Note: Questions 1-2 require filtering/aggregating CSV data. Questions 3-4 require searching unstructured text. Questions 5-6 require combining insights from both sources.*

**What you'll implement:**
- Query router that classifies questions and selects appropriate data source(s)
- CSV retrieval using bash tools (grep, awk, cut) or pandas
- Text retrieval using semantic search or keyword matching
- Result combination strategy for multi-source queries
- Answer generation with the LLM


In [1]:
import os
import json
import logging
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Setup logging
logging.basicConfig(
    format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)
if os.getenv("GROQ_API_KEY"):
    logger.info("GROQ_API_KEY is set")

from langchain_community.chat_models import ChatLiteLLM

# Configure the LLM - using Groq's free Llama model
MODEL_ID = "groq/llama-3.1-8b-instant"
MODEL_ID = "groq/llama-3.3-70b-versatile" 

llm = ChatLiteLLM(
    model=MODEL_ID,
    temperature=0
)
logger.info(f"Using model: {MODEL_ID}")

[2026-03-06 04:09:09,535] p3678 {712715439.py:17} INFO - GROQ_API_KEY is set
/home/ubuntu/dsan6725/a3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_3678/712715439.py:25: LangChainDeprecationWarning: The class `ChatLiteLLM` was deprecated in LangChain 0.3.24 and will be removed in 1.0. An updated version of the class exists in the `langchain-litellm package and should be used instead. To use it run `pip install -U `langchain-litellm` and import as `from `langchain_litellm import ChatLiteLLM``.
  llm = ChatLiteLLM(
[2026-03-06 04:09:21,366] p3678 {712715439.py:29} INFO - Using model: groq/llama-3.3-70b-versatile


In [2]:
"""
QUERY ROUTING
"""
from sre_parse import State
import time
from langgraph.types import Command

def classify_query(query: str) -> str:
    # Structure questions might use tree or find
    # Code search questions might use grep with file type filters
    # Dependency questions should look at pyproject.toml and package.json
    # Documentation questions should search the docs/ folder

    # use LLM to classify the query  
    system_prompt = """
        You are a query classification tool for a data retrieval system.

        The data is structured as follows:

        Structured Data (CSV), located in `../data/structured/daily_sales.csv`:
        - 1000 daily sales records across 90 days (Oct-Dec 2024)
        - 35 products across 10 categories
        - Columns: `date, product_id, product_name, category, units_sold, unit_price, total_revenue, region`
        - date format: YYYY-MM-DD
        - The region column contains values: North, South, East, West, Central

        Unstructured Data (Text), located in `../data/unstructured/`:
        - 10 product pages with detailed descriptions, specifications, and customer reviews
        - Similar to what you'd find on an e-commerce website
        - Includes product features, technical specs, and 5 customer reviews per product

        # Given a user query, determine whether it requires retrieving structured data (CSV), unstructured data (text), or both. If it requires structured data, output 'structured'. If it requires unstructured data, output 'unstructured'. If it could potentially use both, output 'both'.
        
        # Rules:
        - Choose exactly ONE category('structured', 'unstructured', 'both').
        - Output ONLY the category name.
        - Do not explain your answer.
        - Do not include any extra text.

        Classify the following query:
    """

    wait = 1
    for i in range(5):
        time.sleep(wait)
        try:
            response = llm.invoke(f"Prompt: {system_prompt}\nQuestion: {query.lower()}")
            break
        except Exception as e:
            logger.error(f"Error invoking LLM: {e}")
        wait *= 2
    
    return response.content

/tmp/ipykernel_3678/508047581.py:4: DeprecationWarning: module 'sre_parse' is deprecated
  from sre_parse import State


In [ ]:
"""
BASH TOOL RETRIEVAL
"""
import subprocess

def extract_keywords(query):
# Words = re.findall(r"\w+", query.lower())
# Return [w for W in Words if W not in stop_words and len(w) > 3]
    system_prompt = "You are a retrieval augmented generation AI tool that takes in a query and provides a list of keywords that represent the query that can be searched for using 'grep' to find relevant context. Only output the list of words separated by commas. Do not include any other text. Only include FIVE words. Use strings that are likely to be found in a review or product description. Use nouns when possible. Find relevant keywords for the following query: " 

    wait = 1
    for _ in range(5):
        time.sleep(wait)
        try:
            response = llm.invoke(f"Prompt: {system_prompt}\n{query.lower()}")
            break
        except Exception as e:
            logger.error(f"Error invoking LLM: {e}")
        wait *= 2
    
    query_words = response.content
    query_words = [w.strip() for w in query_words.split(",")]
    return query_words

def generate_csv_query(query: str) -> str:
    system_prompt = """
    You are a data query generation tool that takes in a user question and generates a bash/grep command to pull relevant data from a raw csv file ('../data/structured/daily_sales.csv') to answer that question based on the following CSV schema.

    - Columns (in order): `date, product_id, product_name, category, units_sold, unit_price, total_revenue, region`
    - date format: YYYY-MM-DD
    - product_id format: {4 letter category code}{3 digit number}
        - category codes: BEAU, BOOK, CLTH, ELEC, FOOD, HOME, OFFC, PETS, SPRT, TOYS
    - Region column contains values: North, South, East, West, Central

    Output the raw grep command as a single string without any explanation, backticks, or extra text. Do NOT put it in a code block.
    Then, add a detailed description of what the bash command outputs.
    
    The user question is:
    """

    wait = 1
    for _ in range(5):
        time.sleep(wait)
        try:
            response = llm.invoke(f"Prompt: {system_prompt}\n{query.lower()}")
            break
        except Exception as e:
            logger.error(f"Error invoking LLM: {e}")
        wait *= 2

    response_lines = response.content.split("\n")
    command = response_lines[0].strip()
    descrip = " ".join(response_lines[1:]).strip()
    
    return command, descrip

def run_bash(cmd: str, max_output: int) -> str:
    result = subprocess.run(
        cmd,
        shell=True,
        capture_output=True,
        text=True
    )
    output = result.stdout
    return output[:max_output]

def retrieve_context(query: str, q_class: str) -> str:
    max_output = 20000
    file_names = []
    q_class = classify_query(query)
    cmds = []
    context = ""
    if q_class == "structured" or q_class == "both":
        cmd, explanation = generate_csv_query(query)
        cmds.append(cmd)
        context += f"\n### Bash command to retrieve structured data:\n{cmd}\n"
        print(run_bash(cmd, max_output=max_output))
        context += f"\n### Output of the bash command:\n{run_bash(cmd, max_output=max_output)}\n"
        context += f"\n### Explanation of the structured data retrieval:\n{explanation}\n"
    elif q_class == "unstructured" or q_class == "both":
        query_words = extract_keywords(query + context)
        print(query_words)
        for word in query_words:
            cmd = f"grep -R -n -C 10 '{word}' ../data/unstructured"
            cmds.append(cmd)
            context += run_bash(cmd, max_output=max_output//len(query_words))
            file_names.extend(run_bash(cmd + ' -l', max_output=max_output).splitlines())
    
    logger.info(f"\n\nBash command: {cmd}\n")
    logger.info(f"Retrieved context length: {len(context)}\n")
    return [context, cmds, file_names]


In [6]:
import time

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. "
    "\n\n"
)

questions = []
questions.append("What was the total revenue for Electronics category in December 2024?")
questions.append("Which region had the highest sales volume?")
questions.append("What are the key features of the Wireless Bluetooth Headphones?")
questions.append("What do customers say about the Air Fryer's ease of cleaning?")
questions.append("Which product has the best customer reviews and how well is it selling?")
questions.append("I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?")

output_file = "part2_results.txt"
open(output_file, "w").close()

for q in questions:
    wait = 1
    classification = classify_query(q)
    logger.info(f"Classified query as: {classification}")

    context, cmd, file_names = retrieve_context(q, classification)
    # print("\n\n\n")
    print(len(context))
    context = context[:12000]
    # print("\n\n\n")

    for i in range(5):
        time.sleep(wait)
        try:
            response = llm.invoke(
                f"Prompt: {system_prompt}\nContext:\n{context}\n\nQuestion:{q}"
            )
            ans = response.content
            break
        except Exception as e:
            logger.error(f"Error invoking LLM: {e}")

        wait *= 2  

    with open(output_file, "a") as f:
        f.write(f"Question: {q}\n")
        f.write(f"Classification: {classification}\n")  
        f.write(f"Commands: {cmd}\n")
        f.write(f"Context: {context}\n")
        # f.write(f"Files: {', '.join(file_names_list[i])}\n")
        f.write(f"Answer: {ans}\n\n")

04:11:30 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
[2026-03-06 04:11:30,402] p3678 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
04:11:30 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 04:11:30,535] p3678 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-06 04:11:30,538] p3678 {792098260.py:24} INFO - Classified query as: structured
04:11:31 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
[2026-03-06 04:11:31,541] p3678 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
04:11:31 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 04:11:31,621] p3678 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
04:11:32 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM c

142864

984


04:11:34 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
[2026-03-06 04:11:34,682] p3678 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
04:11:34 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 04:11:34,858] p3678 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
04:11:35 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
[2026-03-06 04:11:35,863] p3678 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
04:11:35 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 04:11:35,961] p3678 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-06 04:11:35,963] p3678 {792098260.py:24} INFO - Classified query as: structured
04:11:36 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM c

 28804

1015


04:11:40 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
[2026-03-06 04:11:40,086] p3678 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
04:11:40 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 04:11:40,644] p3678 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
04:11:41 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
[2026-03-06 04:11:41,647] p3678 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
04:11:41 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 04:11:41,724] p3678 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-06 04:11:41,726] p3678 {792098260.py:24} INFO - Classified query as: unstructured
04:11:42 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM

...

['cleaning', 'easy', 'simple', 'maintenance', 'convenient']
4000


04:11:50 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
[2026-03-06 04:11:50,112] p3678 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
04:11:50 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 04:11:50,651] p3678 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
04:11:51 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
[2026-03-06 04:11:51,655] p3678 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
04:11:51 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 04:11:51,766] p3678 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-06 04:11:51,768] p3678 {792098260.py:24} INFO - Classified query as: both
04:11:52 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM complet

2024-12-18,OFFC001,Ergonomic Office Chair,Office Supplies,50,249.99,12499.5,North

1098


04:11:56 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
[2026-03-06 04:11:56,247] p3678 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq


KeyboardInterrupt: 